In [5]:
import json
import os
import sys
from pathlib import Path

def locate_project_root(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'tutorial' / 'inlist_project').is_file():
            return candidate
    raise FileNotFoundError('Could not locate tutorial/inlist_project')

PROJECT_ROOT = locate_project_root()
PIPELINE_DIR = PROJECT_ROOT / 'scripts' / 'mams_profiles' / 'pipeline'
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from mams_pipeline import (
    build_configs, inspect_runs, model_label, prepare_grid,
    render_inlist_project, run_local, submit_slurm,
)

print(f'Project root: {PROJECT_ROOT}')

# Main user-editable parameters
MASSES_MSUN = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.50,
    0.80,
    1.00,
    1.50,
    2.00,
    3.00,
    5.00
]
INITIAL_Y = 0.2703
INITIAL_Z = 0.0142
CENTRAL_H1_LIMIT = 0.35

# Slurm settings used only if SUBMIT_SLURM is enabled.
PARTITION = 'typhon'
WALLTIME = '7-00:00:00'
CPUS_PER_TASK = 1

# An environment override makes validation and alternate output locations easy.
OUTPUT_ROOT = Path(os.environ.get(
    'MAMS_OUTPUT_ROOT', PROJECT_ROOT / 'results' / 'mams_profiles'
)).resolve()
TEMPLATE_DIR = PROJECT_ROOT / 'tutorial'
OVERWRITE = True
PREPARE_RUNS = True
RUN_LOCAL = True       # Runs models sequentially and waits; potentially slow.
SUBMIT_SLURM = False    # Submits all prepared models and returns job IDs.

if RUN_LOCAL and SUBMIT_SLURM:
    raise ValueError('Choose local execution or Slurm submission, not both.')


configs = build_configs(
    MASSES_MSUN,
    initial_y=INITIAL_Y,
    initial_z=INITIAL_Z,
    central_h1_limit=CENTRAL_H1_LIMIT,
    partition=PARTITION,
    walltime=WALLTIME,
    cpus_per_task=CPUS_PER_TASK,
)

for config in configs:
    print(f'{config.mass_msun:8g} M_sun  ->  {model_label(config.mass_msun)}')

template_text = (TEMPLATE_DIR / 'inlist_project').read_text()
preview = render_inlist_project(template_text, configs[0])
keys_to_show = (
    'filename_for_profile_when_terminate', 'save_pulse_data_filename',
    'initial_mass', 'initial_y', 'initial_z', 'Zbase',
    'xa_central_lower_limit(1)',
)
for line in preview.splitlines():
    if any(key in line for key in keys_to_show):
        print(line)

assert f"'{model_label(configs[0].mass_msun)}.profile'" in preview
assert f"'{model_label(configs[0].mass_msun)}.mesa'" in preview
print('In-memory smoke test passed.')

runs = []
if PREPARE_RUNS:
    runs = prepare_grid(
        configs, TEMPLATE_DIR, OUTPUT_ROOT, overwrite=OVERWRITE
    )
    print(f'Prepared {len(runs)} run(s) under {OUTPUT_ROOT}')
else:
    print('PREPARE_RUNS is false; no directories were written.')

inspect_runs(runs)

job_ids = {}
if RUN_LOCAL:
    for run in runs:
        print(f'Running {run.label} ...', flush=True)
        run_local(run)
elif SUBMIT_SLURM:
    for run in runs:
        job_ids[run.label] = submit_slurm(run)
    print(json.dumps(job_ids, indent=2))
else:
    print('Setup only: no MESA processes or Slurm jobs were started.')

status = inspect_runs(runs)
print(json.dumps(status, indent=2))

if runs:
    metadata = json.loads(runs[0].metadata_path.read_text())
    print('\nFirst run metadata:')
    print(json.dumps(metadata, indent=2))

    if runs[0].gyre_path.is_file():
        fields = runs[0].gyre_path.read_text().splitlines()[0].split()
        names = ['zones', 'mass_g', 'radius_cm', 'luminosity_erg_s', 'version']
        print('\nGYRE header:', dict(zip(names, fields)))

Project root: /Users/mchau/mesa_tutorial
     0.1 M_sun  ->  MAMS0p1Msun
    0.15 M_sun  ->  MAMS0p15Msun
     0.2 M_sun  ->  MAMS0p2Msun
    0.25 M_sun  ->  MAMS0p25Msun
     0.3 M_sun  ->  MAMS0p3Msun
    0.35 M_sun  ->  MAMS0p35Msun
     0.4 M_sun  ->  MAMS0p4Msun
     0.5 M_sun  ->  MAMS0p5Msun
     0.8 M_sun  ->  MAMS0p8Msun
       1 M_sun  ->  MAMS1Msun
     1.5 M_sun  ->  MAMS1p5Msun
       2 M_sun  ->  MAMS2Msun
       3 M_sun  ->  MAMS3Msun
       5 M_sun  ->  MAMS5Msun
   filename_for_profile_when_terminate = 'MAMS0p1Msun.profile'
   save_pulse_data_filename = 'MAMS0p1Msun.mesa'
   initial_mass = 0.1
   initial_y = 0.2703  ! Helium mass fraction
   initial_z = 0.0142  ! Metal mass fraction, hydrogen = 1 - metal - helium
   xa_central_lower_limit(1) = 0.35
  use_Zbase_for_Type1 = .true.
  Zbase = 0.0142
In-memory smoke test passed.
Prepared 14 run(s) under /Users/mchau/mesa_tutorial/results/mams_profiles
Running MAMS0p1Msun ...
Running MAMS0p15Msun ...
Running MAMS0p2Msun ...


In [2]:
import os
import sys

print(sys.executable)
print(os.environ.get("MESA_DIR"))

/Users/mchau/mesa_tutorial/.venv/bin/python
/Users/mchau/mesa
